# OpenAI Responses API

OpenAI Responses API from/to translation helpers

In [ ]:
#| default_exp openai_responses

## Imports

In [ ]:
#| export
import json
from collections import Counter
from fastcore.utils import *
from fastcore.meta import *
from fastspec.errors import api_error_from_event

from fastllm.types import *
from fastllm.streaming import *
from fastllm.streaming import mk_acollect_stream

In [ ]:
#| hide
import yaml
from fastspec.spec import *
from fastspec.oapi import *
from fastcore.test import *
from toolslm.funccall import get_schema
from cachy.core import enable_cachy, disable_cachy
from importlib.resources import files

In [ ]:
enable_cachy(hdrs=('content-type',))

In [ ]:
specs_path = files('fastllm') / 'specs'
oai_spec  = SpecParser.from_openapi(dict2obj(yaml.safe_load((specs_path/'openai.with-code-samples.yml').read_text())))
oai_spec

SpecParser(base_url='https://api.openai.com/v1', ops=241)

In [ ]:
oai_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"})

In [ ]:
oai_cli.groups.keys()

dict_keys(['assistants', 'audio', 'batches', 'chat', 'completions', 'containers', 'conversations', 'embeddings', 'evals', 'files', 'fine_tuning', 'images', 'models', 'moderations', 'organization', 'projects', 'realtime', 'responses', 'threads', 'uploads', 'vector_stores', 'videos', 'skills', 'chatkit'])

## Normalize

Helpers to transform API responses into the internal representation types

### Tool Calls

Normalize OpenAI tool calls shape(s)

In [ ]:
def lite_mk_func(f):
    if isinstance(f, dict): return f
    return {'type':'function', 'function':get_schema(f, pname='parameters')}

In [ ]:
def simple_add(a:int,b:int):
    'add numbers'
    return a + b

In [ ]:
sch = lite_mk_func(simple_add);sch

{'type': 'function',
 'function': {'name': 'simple_add',
  'description': 'add numbers',
  'parameters': {'type': 'object',
   'properties': {'a': {'description': '', 'type': 'integer'},
    'b': {'description': '', 'type': 'integer'}},
   'required': ['a', 'b']}}}

In [ ]:
#| export
def norm_tool_call(item):
    typ = item.get("type", "")
    if typ == "function_call":
        extra = {k:v for k,v in item.items() if k not in ("id","name","arguments")}
        return ToolCall(id=item["id"], name=item["name"], arguments=json.loads(item["arguments"]), extra=extra)
    if typ.endswith("_call"):  # web_search_call, file_search_call, etc.
        name = typ.removesuffix("_call")
        extra = {k:v for k,v in item.items() if k not in ("id")}
        return ToolCall(id=item.get("id",""), name=name, arguments=item.get("action", {}), server=True, extra=extra)

def norm_tool_calls(resp):
    "Extract Responses API tool call items as normalized tool calls."
    out = []
    for item in resp.get('output', []):
        if tc := norm_tool_call(item): out.append(tc)
    return out

User defined:

In [ ]:
resp = await oai_cli.responses.create_response(model='gpt-4o-mini', input="What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.", tools=[{"type": "function", **sch['function']}])
norm_tool_calls(resp)

[ToolCall(id='fc_087e17e37403a89e006a2c03c950e8819fbf92fc030e920598', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_3KwI1KzOFjNYcacoMXZiVdYO'}),
 ToolCall(id='fc_087e17e37403a89e006a2c03c950f8819f8de279776dc22fcc', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_HyRRnjaXVvxfI0SrPtwFP2Zk'})]

Web search:

In [ ]:
resp = await oai_cli.responses.create_response(model='gpt-4o-mini', input="What is the weather in San Francisco today?", tools=[{"type": "web_search_preview"}])
norm_tool_calls(resp)

[ToolCall(id='ws_006db67bf97d1b0d006a2c03ca4c3c81a1b4b723c262bafc74', name='web_search', arguments={'type': 'search', 'queries': ['San Francisco weather today'], 'query': 'San Francisco weather today'}, server=True, extra={'type': 'web_search_call', 'status': 'completed', 'action': {'type': 'search', 'queries': ['San Francisco weather today'], 'query': 'San Francisco weather today'}})]

File vector search:

In [ ]:
norm_tool_calls({"output": [{"type": "file_search_call", "id": "fs_abc123", "status": "completed", "queries": ["quarterly revenue"], "results": [{"file_id": "file-xyz", "text": "Revenue was $1M"}]}]})

[ToolCall(id='fs_abc123', name='file_search', arguments={}, server=True, extra={'type': 'file_search_call', 'status': 'completed', 'queries': ['quarterly revenue'], 'results': [{'file_id': 'file-xyz', 'text': 'Revenue was $1M'}]})]

MCP:

In [ ]:
norm_tool_calls({"output": [{"type": "function_call", "call_id": "call_mcp1", "id": "fc_mcp1", "name": "mcp__weather__get_forecast", "arguments": '{"city": "Istanbul"}', "status": "completed"}]})

[ToolCall(id='fc_mcp1', name='mcp__weather__get_forecast', arguments={'city': 'Istanbul'}, server=False, extra={'type': 'function_call', 'call_id': 'call_mcp1', 'status': 'completed'})]

### Usage

Normalize OpenAI usage shape(s)

In [ ]:
#| export
def norm_usage(resp):
    "Normalize OpenAI usage shape(s)."
    if not (usg:=resp.get("usage")): return None
    pt = int(usg.get("prompt_tokens", usg.get("input_tokens", 0)) or 0)
    ct = int(usg.get("completion_tokens", usg.get("output_tokens", 0)) or 0)
    tt = int(usg.get("total_tokens", pt + ct) or (pt + ct))
    pd = usg.get("prompt_tokens_details") or usg.get("input_tokens_details") or {}
    cd = usg.get("completion_tokens_details") or usg.get("output_tokens_details") or {}
    cached = int(pd.get("cached_tokens", 0) or 0)
    reasoning = int(cd.get("reasoning_tokens", 0) or 0)
    server_tool_use = dict(Counter(o['type'] for o in resp.get('output', []) if o.get('type') != 'function_call' and o.get('type', '').endswith('_call')))
    if server_tool_use: usg['server_tool_use'] = server_tool_use
    return Usage(prompt_tokens=pt, completion_tokens=ct, total_tokens=tt,
                 cached_tokens=cached, reasoning_tokens=reasoning, raw=usg)


Text response:

In [ ]:
resp = await oai_cli.responses.create_response(model='gpt-4o-mini', input=[{"role": "user", "content": "Say hi"}])
norm_usage(resp)

Usage(prompt_tokens=9, completion_tokens=11, total_tokens=20, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 9, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 11, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 20})

Tool call response:

In [ ]:
resp = await oai_cli.responses.create_response(model='gpt-4o-mini', input="What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.", tools=[{"type": "function", **sch['function']}])
norm_usage(resp)

Usage(prompt_tokens=60, completion_tokens=53, total_tokens=113, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 60, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 53, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 113})

Server tools:

In [ ]:
resp = await oai_cli.responses.create_response(model='gpt-4o-mini', input="What is the weather in San Francisco today?", tools=[{"type": "web_search_preview"}])
norm_usage(resp)

Usage(prompt_tokens=310, completion_tokens=621, total_tokens=931, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 310, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 621, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 931, 'server_tool_use': {'web_search_call': 1}})

### Finish Reason

Normalize OpenAI finish reason shape(s)

In [ ]:
#| export
def norm_finish(resp, tcs=None):
    "Canonicalize finish_reason to OpenAI Chat values: stop, tool_calls, length, content_filter."
    reason = resp.get("status")
    mp = dict(completed=FinishReason.stop, incomplete=FinishReason.length, failed=FinishReason.content_filter)
    r =  mp.get(reason, reason)
    return FinishReason.tool_calls if r==FinishReason.stop and any(~L(tcs).attrgot('server')) else r 

In [ ]:
test_eq(norm_finish(dict(status='completed'), ToolCall('123', 'f', {})), 'tool_calls')
test_eq(norm_finish(dict(status='completed'), ToolCall('123', 'web_search', {}, server=True)), 'stop')

### Non-stream Completion

Normalize OpenAI Responses API object into Completion.

**NB:** Per the OpenAI spec, `text`, `refusal`, and `summary_text` fields are all **required** — the `.get(k, "")` fallbacks are purely defensive against malformed responses. `OutputMessageContent` is a discriminated `oneOf` with exactly two variants: `OutputTextContent` (`output_text`) and `RefusalContent` (`refusal`) — no other content types exist in message blocks.

`summary` is an array of `SummaryTextContent` only — no discriminated union, just one type. So the `if s.get("type") == "summary_text"` check is technically redundant (it'll always be `summary_text`), and `text` is required.

Additionally, `Response.output`, `OutputMessage.content`, and `ReasoningItem.summary` are all **required** fields — so `.get()` with fallback defaults is defensive but not strictly necessary.


In [ ]:
#| export
def norm_parts(resp):
    parts = []
    for item in resp["output"]:
        if (typ:=item["type"]) == "message":
            for c in item["content"]:
                if (ctyp := c["type"]) == "output_text":
                    c['citations'] = c.pop('annotations', [])
                    parts.append(Part(type=PartType.text, text=c['text'], data=c))
                elif ctyp == "refusal":
                    parts.append(Part(type=PartType.refusal, text=c['refusal'], data=c))
        elif typ == "reasoning":
            for s in item["summary"]: parts.append(Part(type=PartType.thinking, text=s['text'], data=item))
        elif typ == "function_call" or typ.endswith("_call"):
            tc = norm_tool_call(item)
            tdata = {**tc.extra, 'id':tc.id, 'name':tc.name, 'arguments':tc.arguments, 'server':tc.server}
            parts.append(Part(type=PartType.tool_use, data=tdata))
    return parts

Users can define `norm_tool_calls`, `norm_parts`, `norm_finish` and `norm_usage` for a given provider, and `mk_completion` will be used to create the canonical `Completion` object based on those functions:

In [ ]:
norm_funcs = dict(norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage)
api_registry.register('openai', **norm_funcs)

#### Text

In [ ]:
mn,api_name,vnd_nm = 'gpt-4o-mini', 'openai', 'openai'
resp = await oai_cli.responses.create_response(model=mn, input="Say hi")
c = mk_completion(resp, mn, api_name, vnd_nm)
c.message.content, c.finish_reason

([Part(type=<PartType.text: 'text'>, text='Hi there! How can I assist you today?', data={'type': 'output_text', 'logprobs': [], 'text': 'Hi there! How can I assist you today?', 'citations': []})],
 <finish_reason.stop: 'stop'>)

#### Tool call

In [ ]:
resp = await oai_cli.responses.create_response(model=mn, input="What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.", tools=[{"type": "function", **sch['function']}])
c = mk_completion(resp, mn, api_name, vnd_nm)
test_eq(c.finish_reason, 'tool_calls')
test_eq(len(c.message.content), 2)
c.tool_calls, c.message.content

([ToolCall(id='fc_087e17e37403a89e006a2c03c950e8819fbf92fc030e920598', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_3KwI1KzOFjNYcacoMXZiVdYO'}),
  ToolCall(id='fc_087e17e37403a89e006a2c03c950f8819f8de279776dc22fcc', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_HyRRnjaXVvxfI0SrPtwFP2Zk'})],
 [Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'type': 'function_call', 'status': 'completed', 'call_id': 'call_3KwI1KzOFjNYcacoMXZiVdYO', 'id': 'fc_087e17e37403a89e006a2c03c950e8819fbf92fc030e920598', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False}),
  Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'type': 'function_call', 'status': 'completed', 'call_id': 'call_HyRRnjaXVvxfI0SrPtwFP2Zk', 'id': 'fc_087e17e37403a89e006a2c03c950f8819f8de279776dc22fcc', 'name': 'simple_add', 'ar

Web search response:

In [ ]:
resp = await oai_cli.responses.create_response(model=mn, input="What is the weather in Istanbul today?", tools=[{"type": "web_search_preview"}])
c = mk_completion(resp,mn,api_name,vnd_nm)
c.tool_calls, next(p.text for p in c.message.content if p.type == 'text')[:100]

([ToolCall(id='ws_0146f4b2e037c07d006a2c03d1a7cc81a28860cbc2165ab41f', name='web_search', arguments={'type': 'search', 'queries': ['Istanbul weather today'], 'query': 'Istanbul weather today'}, server=True, extra={'type': 'web_search_call', 'status': 'completed', 'action': {'type': 'search', 'queries': ['Istanbul weather today'], 'query': 'Istanbul weather today'}})],
 'As of 4:04\u202fPM local time on June 12, 2026, in Istanbul, the weather is sunny with a temperature of 8')

In [ ]:
test_eq(c.tool_calls[0].server, True)
test_eq(c.finish_reason, 'stop')

Refusal response (mocked — hard to trigger reliably via API):

In [ ]:
c = mk_completion({"model": "gpt-4o-mini", "status": "completed", "output": [{"type": "message", "role": "assistant", "content": [{"type": "refusal", "refusal": "I can't help with that request."}]}], "usage": {"input_tokens": 10, "output_tokens": 5, "total_tokens": 15}}, mn, api_name, vnd_nm)
c.message.content

[Part(type=<PartType.refusal: 'refusal'>, text="I can't help with that request.", data={'type': 'refusal', 'refusal': "I can't help with that request."})]

### Streaming

Streaming module implements `PartAccum` and `acollect_stream` which are used to collate canonical `Delta` objects into a final `Completion` object similar to the non-streaming path while yielding text/thinking/citations. Each provider needs to implement `norm_sse_event` which will transform an event into a compatible `Delta` object which will be consumed by the streaming module

In [ ]:
class PrintStream:
    max_print = 10
    cnt = 0
    def print_stream(self,o,postproc=noop):
        if not isinstance(o, (Completion,Part)): 
            if o.get('thinking') and self.cnt<self.max_print: print('🧠',end='',flush=True)
            if (txt:=o.get('text')): print(txt,end='',flush=True)
            if citations:= o.get('citations'): print(postproc(citations),end='',flush=True)
            self.cnt+=1
        else: self.cnt = 0

In [ ]:
print_stream = PrintStream().print_stream

OpenAI responses api returns the whole tool call response with a `response.output_item.done` event which doesn't require argument collation and makes our job easy. If api wasn't returning the collated tool call, we'd have needed to provide delta arguments in Delta object's `tool_call` like:

```py
ToolCall(id=tc.get("id",""), name=fn.get("name",""), arguments={'_delta': fn.get("arguments")}, extra=extra)
```

In [ ]:
#| export
def norm_sse_event(ev, **kwargs):
    "Normalize OpenAI Responses API stream event into Delta."
    ev = obj2dict(ev)
    typ = ev.get("type")
    if typ == "response.output_text.delta":            return Delta(text=ev.get("delta"), raw=ev, **kwargs)
    if typ == "response.reasoning_text.delta":         return Delta(thinking=ev.get("delta",""), raw=ev, **kwargs)
    if typ == "response.reasoning_summary_text.delta": return Delta(thinking=ev.get("delta",""), raw=ev, **kwargs)
    if typ == "response.output_text.annotation.added": return Delta(citations=[ev.get("annotation")], raw=ev, **kwargs)
    if typ == "response.output_item.done":
        item = ev.get("item", {})
        itype = item.get("type", "")
        if itype == "function_call" or itype.endswith("_call"):
            return Delta(tool_calls=[norm_tool_call(item)], raw=ev, **kwargs)
    if typ in ("response.completed", "response.incomplete"):
        resp = ev.get("response", {})
        return Delta(finish_reason=norm_finish(resp), usage=norm_usage(resp), raw=ev, **kwargs)
    if typ == "error": raise api_error_from_event(ev)

Each provider also needs to implement `delta_index_fn` which returns a current idx and last idx and used by `PartAccum` like `part_accum.append(typ, idx, **tc_kwargs)`.

Here `d` is a `Delta` object, `typ` is the name of the event, it also takes last event's name and it's index to decide how to do collation

In [ ]:
#| export
def delta_index_fn(d, typ, last_typ, last_idx):
    'Returns accumulation index for current delta and updated last idx'
    idx = nested_idx(d, 'raw', 'output_index')
    return idx, idx

Now we can create and register openai responses api's `acollect_stream`:

In [ ]:
acollect_stream = partial(mk_acollect_stream, index_fn=delta_index_fn, api_name='openai')

In [ ]:
#| export
@delegates(mk_acollect_stream, but=['index_fn', 'api_name'])
async def acollect_stream(resp, **kwargs):
    res = mk_acollect_stream(norm_and_yield(resp, norm_sse_event), index_fn=delta_index_fn, api_name='openai', **kwargs)
    async for o in res: yield o

In [ ]:
norm_funcs = dict(norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage, acollect_stream=acollect_stream)
api_registry.register('openai', **norm_funcs)

In [ ]:
api_registry.apis['openai']

namespace(finalize_usage=<function fastcore.imports.noop(x=None, *args, **kwargs)>,
          norm_tool_calls=<function __main__.norm_tool_calls(resp)>,
          norm_parts=<function __main__.norm_parts(resp)>,
          norm_finish=<function __main__.norm_finish(resp, tcs=None)>,
          norm_usage=<function __main__.norm_usage(resp)>,
          acollect_stream=<function __main__.acollect_stream(resp, *, model=None, vendor_name=None, stop_callables=None)>)

The following tests demonstrate completions with streaming and non-streaming paths

#### Text

In [ ]:
mn,inp = 'gpt-4o-mini','Hi!'
resp = await oai_cli.responses.create_response(model=mn,input=inp)
comp = mk_completion(resp, mn, api_name, vnd_nm); comp

<div class="prose" markdown="1">

Hi there! How can I assist you today?

<details markdown='1'>

- model: `gpt-4o-mini`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=9, completion_tokens=11, total_tokens=20, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 9, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 11, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 20})`

</details>

</div>

In [ ]:
resp = await oai_cli.responses.create_response(model=mn, input=inp, stream=True)
async for o in acollect_stream(resp, vendor_name=vnd_nm): print_stream(o)

Hello

!

 How

 can

 I

 assist

 you

 today

?

In [ ]:
resp = await oai_cli.responses.create_response(model=mn, input=inp, stream=True)
async for o in acollect_stream(resp, vendor_name=vnd_nm): print_stream(o)

Hello

!

 How

 can

 I

 assist

 you

 today

?

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)
comp.usage, o.usage

#### Thinking

In [ ]:
mn,inp,eff = 'o4-mini','What is 31231231 * 12312?',{"effort": "low", "summary": "auto"}
resp = await oai_cli.responses.create_response(model=mn,input=inp,reasoning=eff)
comp = mk_completion(resp, mn, api_name, vnd_nm)

In [ ]:
comp

<div class="prose" markdown="1">

<details><summary>Thinking</summary>

**Calculating multiplication step-by-step**

The user asked me to multiply 31,231,231 by 12,312, so I'm working through it step by step. I'll split 12,312 into 12,000 and 312 for easier calculation. 

First, I calculate 31,231,231 times 12,000, then I tackle the 31,231,231 times 312. 

After working through the additions and carefully summing them up, I arrive at the final result: 384,518,916,072. Looks like I got it right!

</details>

<details><summary>Thinking</summary>

**Verifying multiplication accuracy**

I'm going to double-check my multiplication using a more direct approach. It looks like 31 million times 12,000 gives around 372 billion, while I calculated 384.5 billion, which is a bit high. However, I've done the breakdown for 12,312 into 12,000 and 312 already.

To verify, I'll divide my result by 12,312 to see if I get 31,231,231. I'll also check the sum of the digits. Everything checks out, and the product is consistent, which means I can confidently say the result is 384,518,916,072.

</details>

31231231 × 12312 = 384,518,916,072

<details markdown='1'>

- model: `o4-mini`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=17, completion_tokens=881, total_tokens=898, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=832, raw={'input_tokens': 17, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 881, 'output_tokens_details': {'reasoning_tokens': 832}, 'total_tokens': 898})`

</details>

</div>

In [ ]:
resp = await oai_cli.responses.create_response(model=mn,input=inp,reasoning=eff,stream=True)
async for o in acollect_stream(resp, vendor_name=vnd_nm): print_stream(o)

🧠

🧠

🧠

🧠

🧠

...

518

,

916

,

072

In [ ]:
test_eq('thinking' in L(comp.message.content).attrgot('type'), True)
test_eq('thinking' in L(o.message.content).attrgot('type'), True)

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)

#### Tool call

In [ ]:
mn,inp = 'gpt-4o-mini','What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'
tools=[{"type": "function", **sch['function']}]
resp = await oai_cli.responses.create_response(model=mn,input=inp,tools=tools)
comp = mk_completion(resp, mn, api_name, vnd_nm)

In [ ]:
comp

<div class="prose" markdown="1">



🔧 simple_add({'a': 3, 'b': 5})



🔧 simple_add({'a': 10, 'b': 20})


<details markdown='1'>

- model: `gpt-4o-mini`
- finish_reason: `tool_calls`
- usage: `Usage(prompt_tokens=60, completion_tokens=53, total_tokens=113, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 60, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 53, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 113})`

</details>

</div>

In [ ]:
resp = await oai_cli.responses.create_response(model=mn, input=inp, tools=tools, stream=True)
async for o in acollect_stream(resp, vendor_name=vnd_nm): print_stream(o)

In [ ]:
def test_tcs(tcs1,tcs2):
    for tc1,tc2 in zip(tcs1,tcs2): 
        test_eq(tc1.name,tc2.name); test_eq(tc1.arguments,tc2.arguments); test_eq(tc1.server,tc2.server)

In [ ]:
test_tcs(comp.tool_calls,o.tool_calls)

In [ ]:
test_eq(len(comp.message.content), len(o.message.content))
test_eq(set(comp.message.content[0].data.keys()), set(o.message.content[0].data.keys()))
test_eq(comp.finish_reason, o.finish_reason)
comp.usage, o.usage

#### Web search

In [ ]:
mn,inp,tools = 'gpt-4o-mini','What is the weather in Istanbul today?',[{"type": "web_search_preview"}]
resp = await oai_cli.responses.create_response(model=mn,input=inp,tools=tools)
comp = mk_completion(resp, mn, api_name, vnd_nm)

In [ ]:
comp

<div class="prose" markdown="1">

As of 4:04 PM local time on June 12, 2026, in Istanbul, the weather is sunny with a temperature of 80°F (27°C).

## Weather for Hastane, Hadımköy-İstanbul Caddesi 34, 34555 Arnavutköy/İstanbul, Türkiye:
Current Conditions: Sunny, 80°F (27°C)

Hourly Forecast:
* 5:00 PM: 80°F (26°C), Sunny
* 6:00 PM: 78°F (25°C), Sunny
* 7:00 PM: 74°F (23°C), Sunny
* 8:00 PM: 72°F (22°C), Sunny
* 9:00 PM: 70°F (21°C), Clear
* 10:00 PM: 68°F (20°C), Clear
* 11:00 PM: 68°F (20°C), Mostly clear
* 12:00 AM: 67°F (19°C), Mostly clear
* 1:00 AM: 67°F (19°C), Partly cloudy
* 2:00 AM: 66°F (19°C), Intermittent clouds
* 3:00 AM: 65°F (18°C), Cloudy
* 4:00 AM: 65°F (18°C), Cloudy


In June, Istanbul typically experiences daytime temperatures around 77°F (25°C) and nighttime lows near 60°F (16°C). The city enjoys approximately 11 hours of sunshine per day, with an average of 6 days of rainfall throughout the month. ([weather2travel.com](https://www.weather2travel.com/turkey/istanbul/june/?utm_source=openai))

Please note that weather conditions can change, so it's advisable to check for the latest updates if you have specific plans. 

🔧 web_search({'type': 'search', 'queries': ['Istanbul weather today'], 'query': 'Istanbul weather today'})


<details markdown='1'>

- model: `gpt-4o-mini`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=309, completion_tokens=416, total_tokens=725, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 309, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 416, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 725, 'server_tool_use': {'web_search_call': 1}})`

</details>

</div>

Same with streaming:

In [ ]:
resp = await oai_cli.responses.create_response(model=mn, input=inp, tools=tools, stream=True)
async for o in acollect_stream(resp, vendor_name=vnd_nm): print_stream(o)

As

 of

4

:

31

...

 the

 month

.

([weather2visit.com](https://www.weather2visit.com/middle-east/turkey/istanbul-june.htm?utm_source=openai))

[{'type': 'url_citation', 'end_index': 1072, 'start_index': 965, 'title': 'İstanbul Weather in June 2026 | Turkey Averages | Weather-2-Visit', 'url': 'https://www.weather2visit.com/middle-east/turkey/istanbul-june.htm?utm_source=openai'}]

In [ ]:
cites = o.message.content[-1].data['citations']; cites

[{'type': 'url_citation',
  'end_index': 1072,
  'start_index': 965,
  'title': 'İstanbul Weather in June 2026 | Turkey Averages | Weather-2-Visit',
  'url': 'https://www.weather2visit.com/middle-east/turkey/istanbul-june.htm?utm_source=openai'}]

In [ ]:
o.message.content[-1].text[cites[0]['start_index']:]

'([weather2visit.com](https://www.weather2visit.com/middle-east/turkey/istanbul-june.htm?utm_source=openai)) '

In [ ]:
test_tcs(comp.tool_calls, o.tool_calls)
test_eq(comp.finish_reason, o.finish_reason)
comp.usage, o.usage

(Usage(prompt_tokens=309, completion_tokens=416, total_tokens=725, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 309, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 416, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 725, 'server_tool_use': {'web_search_call': 1}}),
 Usage(prompt_tokens=309, completion_tokens=413, total_tokens=722, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 309, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 413, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 722, 'server_tool_use': {'web_search_call': 1}}))

## Denormalize

Helpers to transform internal representation types back to API payload

### Msgs

In [ ]:
def mk_user_msg(txt): return Msg(role='user', content=[Part(type=PartType.text, text=txt)])

Helpers to translate back to provider specific inputs from our internal `Msg` representation

In [ ]:
mn,inp = 'gpt-4o-mini','What is 3 + 5 and 10 + 20. Use the simple_add tool in parallel'
tools=[{"type": "function", **sch['function']}]
resp = await oai_cli.responses.create_response(model=mn, input=inp, tools=tools, stream=True)
async for comp in acollect_stream(resp, vendor_name=vnd_nm): pass

In [ ]:
comp

<div class="prose" markdown="1">



🔧 simple_add({'a': 3, 'b': 5})



🔧 simple_add({'a': 10, 'b': 20})


<details markdown='1'>

- model: `None`
- finish_reason: `tool_calls`
- usage: `Usage(prompt_tokens=59, completion_tokens=53, total_tokens=112, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 59, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 53, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 112})`

</details>

</div>

In [ ]:
trmsg = mk_tool_res_msg(comp.tool_calls, ('8', '30')); trmsg

<div class="prose" markdown="1">

**Msg**

- role: `tool`

<contents>

**Part** (`tool_result`)

8

<details markdown='1'>

- data: `{'id': 'fc_014e405221c065e3006a2c0a44850481a0a8aa3f863067283d', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False}`

</details>

**Part** (`tool_result`)

30

<details markdown='1'>

- data: `{'id': 'fc_014e405221c065e3006a2c0a44852081a0896938c5c8fb3ac3', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False}`

</details>

</contents>

</div>

In [ ]:
#| export
def _sanid(id_str): return id_str[:64] # codex max 64 char limit

In [ ]:
#| export
def denorm_tool_use(p:Part):
    "Convert canonical tool_use Part back to OpenAI Responses function_call item."
    return dict(type='function_call', call_id=_sanid(p.data.get('id')), name=p.data.get('name'), arguments=json.dumps(p.data.get('arguments', '{}')))

In [ ]:
L(comp.message.content).map(denorm_tool_use)

[{'type': 'function_call', 'call_id': 'fc_014e405221c065e3006a2c0a44850481a0a8aa3f863067283d', 'name': 'simple_add', 'arguments': '{"a": 3, "b": 5}'}, {'type': 'function_call', 'call_id': 'fc_014e405221c065e3006a2c0a44852081a0896938c5c8fb3ac3', 'name': 'simple_add', 'arguments': '{"a": 10, "b": 20}'}]

This is not exported -- media tool results are added later

In [ ]:
def denorm_tool_result(p:Part):
    "Convert canonical tool result back to OpenAI Responses function_call_output item."
    return dict(type='function_call_output', call_id=_sanid(p.data.get('id')), output=str(p.text))

In [ ]:
#| export
def denorm_tool(m:Msg):
    items = []
    for part in m.content:
        if part.type == PartType.tool_result: items.append(denorm_tool_result(part))
    return items

In [ ]:
denorm_tool(trmsg)

[{'type': 'function_call_output',
  'call_id': 'fc_014e405221c065e3006a2c0a44850481a0a8aa3f863067283d',
  'output': '8'},
 {'type': 'function_call_output',
  'call_id': 'fc_014e405221c065e3006a2c0a44852081a0896938c5c8fb3ac3',
  'output': '30'}]

This is not exported -- media inputs are added later

In [ ]:
def denorm_user(m:Msg):
    "Convert canonical user Msg to OpenAI Responses input message."
    parts = [{"type": "input_text", "text": p.text or ""} for p in m.content if p.type == PartType.text]
    return dict(type="message", role="user", content=parts)

In [ ]:
umsg = mk_user_msg(inp)
denorm_user(umsg)

{'type': 'message',
 'role': 'user',
 'content': [{'type': 'input_text',
   'text': 'What is 3 + 5 and 10 + 20. Use the simple_add tool in parallel'}]}

In [ ]:
#| export
def denorm_assistant(m:Msg):
    items, srv_call_id = [], None
    for p in m.content:
        if p.type == PartType.tool_use:
            items.append(denorm_tool_use(p))
            # TODO: Server tcs are excluded during deserialization in lisette, so this
            # is not used. Anthropic checks for `srvtoolu` and requires the full tool result metadata
            # which is a lot to store.
            # Instead of reconstructing server tool calls as actual tool uses we can make it a regular msg
            # telling AI a server tool was executed
            if p.data.get('server'):
                srv_txt = f"[Server tool `{p.data['name']}` executed successfully, results are generated below]"
                items.append(dict(type='function_call_output', call_id=p.data['id'], output=srv_txt))    
        elif p.type in (PartType.text, PartType.thinking):
            items.append(dict(type="message", role="assistant", content=[dict(type="output_text", text=p.text)]))
    return items

In [ ]:
denorm_assistant(comp.message)

[{'type': 'function_call',
  'call_id': 'fc_014e405221c065e3006a2c0a44850481a0a8aa3f863067283d',
  'name': 'simple_add',
  'arguments': '{"a": 3, "b": 5}'},
 {'type': 'function_call',
  'call_id': 'fc_014e405221c065e3006a2c0a44852081a0896938c5c8fb3ac3',
  'name': 'simple_add',
  'arguments': '{"a": 10, "b": 20}'}]

In [ ]:
#| export
def denorm_msgs(msgs:list[Msg]):
    res = []
    for m in msgs:
        if m.role == 'user':        res.append(denorm_user(m))
        elif m.role == 'assistant': res.extend(denorm_assistant(m))
        elif m.role == 'tool':      res.extend(denorm_tool(m))
    return res

In [ ]:
inputs = denorm_msgs([umsg, comp.message, trmsg]); inputs

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text',
    'text': 'What is 3 + 5 and 10 + 20. Use the simple_add tool in parallel'}]},
 {'type': 'function_call',
  'call_id': 'fc_014e405221c065e3006a2c0a44850481a0a8aa3f863067283d',
  'name': 'simple_add',
  'arguments': '{"a": 3, "b": 5}'},
 {'type': 'function_call',
  'call_id': 'fc_014e405221c065e3006a2c0a44852081a0896938c5c8fb3ac3',
  'name': 'simple_add',
  'arguments': '{"a": 10, "b": 20}'},
 {'type': 'function_call_output',
  'call_id': 'fc_014e405221c065e3006a2c0a44850481a0a8aa3f863067283d',
  'output': '8'},
 {'type': 'function_call_output',
  'call_id': 'fc_014e405221c065e3006a2c0a44852081a0896938c5c8fb3ac3',
  'output': '30'}]

In [ ]:
resp = await oai_cli.responses.create_response(model=mn, input=inputs, tools=tools, stream=True)
async for comp in acollect_stream(resp, vendor_name=vnd_nm): pass
comp

<div class="prose" markdown="1">

The results are:

- \(3 + 5 = 8\)
- \(10 + 20 = 30\)

<details markdown='1'>

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=115, completion_tokens=28, total_tokens=143, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 115, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 28, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 143})`

</details>

</div>

### Tool Schemas

In [ ]:
#| export
def denorm_tool_schs(tools):
    "Convert canonical tools to OpenAI Responses format."
    out = []
    for t in tools:
        fn = fn_schema(t)
        if fn is None: out.append(t); continue
        name, desc, params = fn
        out.append(dict(type='function', name=name, description=desc, parameters=params))
    return out

In [ ]:
test_eq(denorm_tool_schs([sch]), [{'type': 'function', 'name': 'simple_add', 'description': 'add numbers', 'parameters': sch['function']['parameters']}])
test_eq(denorm_tool_schs([{"type": "web_search_preview"}]), [{"type": "web_search_preview"}])

### Tool Choice

In [ ]:
#| export
def denorm_tool_choice(v):
    "Map canonical tool_choice to OpenAI Responses format."
    _tc_modes = {'auto', 'required', 'any', 'force', 'none', 'off', 'disabled'}
    if v is None: return None
    if v in _tc_modes: return v if v in ('auto','none','required') else {'auto':'auto','any':'required','force':'required','off':'none','disabled':'none'}[v]
    return {'type': 'function', 'name': v}

Modes

In [ ]:
test_eq(denorm_tool_choice('auto'), 'auto')
test_eq(denorm_tool_choice('required'), 'required')

Tool name

In [ ]:
test_eq(denorm_tool_choice('simple_add'), {'type': 'function', 'name': 'simple_add'})

None

In [ ]:
test_eq(denorm_tool_choice(None), None)

### Reasoning Effort

No `disable` option maybe add later if needed. Anthropic uses the newer adaptive thinking which will error with legacy models < 4.6

In [ ]:
#| export
def denorm_reasoning(v):
    "Map canonical reasoning_effort to OpenAI Responses reasoning param."
    if v is None: return None
    return {'effort': v} if isinstance(v, str) else v

In [ ]:
test_eq(denorm_reasoning('low'), {'effort': 'low'})
test_eq(denorm_reasoning(None), None)

### Web Search Options

In [ ]:
#| export
def denorm_web_search(v):
    "Map canonical web_search_options to OpenAI Responses web_search_preview tool."
    t = {"type": "web_search_preview"}
    if (typ := (v or {}).get("type")): t["type"] = typ
    if (s := (v or {}).get("search_context_size")): t["search_context_size"] = s
    if (u := (v or {}).get("user_location")): t["user_location"] = u
    return t

None / empty

In [ ]:
test_eq(denorm_web_search(None), {"type": "web_search_preview"})

user_location

In [ ]:
loc = {"approximate": {"city": "Istanbul"}}
test_eq(denorm_web_search({"user_location": loc})["user_location"], loc)

### System

In [ ]:
#| export
def denorm_system(sp): return sys_text(part_txt(sp))

In [ ]:
sp = 'You are a pirate. Always respond in pirate speak. Keep it to one sentence.'
msg1 = mk_user_msg('What are you?')

OpenAI responses pass sp to `instructions`

In [ ]:
resp = await oai_cli.responses.create_response(model='gpt-4o-mini', input=denorm_msgs([msg1]), instructions=denorm_system(sp), stream=True)
async for comp in acollect_stream(resp, vendor_name=vnd_nm): pass
comp

<div class="prose" markdown="1">

I be a scallywag of the seas, lookin' for treasure and adventure, savvy?

<details markdown='1'>

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=32, completion_tokens=21, total_tokens=53, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 32, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 21, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 53})`

</details>

</div>

### Media Inputs

`fastllm` canonicalizes multimodal inputs into `Part(type, text, data)` where `text` carries the URL or data URL, and `data` is reserved for optional metadata. Each provider's denorm layer converts this canonical form into the provider's wire format.

**Canonical Part Types**

| `Part.type` | `Part.text` | Description |
|---|---|---|
| `input_image` | URL or `data:image/...;base64,...` | Image input — JPEG, PNG, GIF, WEBP |
| `input_audio` | `data:audio/...;base64,...` | Audio input — WAV, MP3 (base64 only for OpenAI) |
| `input_video` | URL | Video input — MP4, WebM (Gemini only) |
| `input_file` | URL or `data:application/...;base64,...` | Document input — PDF, DOCX, TXT, etc. |

**Provider Wire Formats**

**OpenAI Responses** ([InputImageContent](https://developers.openai.com/api/reference/resources/responses/methods/create), [InputFileContent](https://developers.openai.com/api/reference/resources/responses/methods/create) — `specs/openai.with-code-samples.yml:68361,68429`):
- Image: `{"type": "input_image", "image_url": url_or_data_url}`
- File: `{"type": "input_file", "file_url": url}` or `{"type": "input_file", "file_data": data_url, "filename": "..."}`
- Audio/Video: not supported

**OpenAI Chat** ([ContentPartImage](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create), [ContentPartAudio](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create), [ContentPartFile](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create) — `specs/openai.with-code-samples.yml:35185,35217,35253`):
- Image: `{"type": "image_url", "image_url": {"url": url_or_data_url}}`
- Audio: `{"type": "input_audio", "input_audio": {"data": b64, "format": "wav"|"mp3"}}` — base64 only, WAV/MP3 only
- File: `{"type": "file", "file": {"file_data": data_url, "filename": "..."}}` — base64/file_id only, no URL
- Video: not supported

**Anthropic** ([RequestImageBlock](https://docs.anthropic.com/en/api/messages), [RequestDocumentBlock](https://docs.anthropic.com/en/api/messages) — `specs/anthropic.yml:12159,6740`):
- Image: `{"type": "image", "source": {"type": "url", "url": ...}}` or `{"type": "image", "source": {"type": "base64", "media_type": mime, "data": b64}}`
- File: `{"type": "document", "source": ...}` — same source pattern, PDF only (`media_type: "application/pdf"`)
- Audio/Video: not supported
- Supports `cache_control` on all content blocks

**Gemini** ([Part/Blob/FileData](https://ai.google.dev/api/generate-content) — `specs/gemini.json:189–387`):
- All media: `{"inlineData": {"mimeType": mime, "data": b64}}` or `{"fileData": {"mimeType": mime, "fileUri": url}}`
- Broadest format support — images, audio, video (incl. YouTube URLs), documents
- Supports `videoMetadata` for video start/end offsets

**Provider Support Matrix**

| Canonical type | OpenAI Responses | OpenAI Chat | Anthropic | Gemini |
|---|---|---|---|---|
| `input_image` (URL) | ✅ | ✅ | ✅ | ✅ |
| `input_image` (base64) | ✅ | ✅ | ✅ | ✅ |
| `input_audio` (base64) | ❌ | ✅ (WAV/MP3) | ❌ | ✅ |
| `input_video` (URL) | ❌ | ❌ | ❌ | ✅ |
| `input_file` (URL) | ✅ | ❌ | ✅ | ✅ |
| `input_file` (base64) | ✅ | ✅ | ✅ | ✅ |


In [ ]:
#| export
def denorm_user(m:Msg):
    "Convert canonical user Msg to OpenAI Responses input message."
    parts = []
    for p in m.content:
        if   p.type == PartType.text:        parts.append({"type": "input_text", "text": p.text or ""})
        elif p.type == PartType.input_image: parts.append(denorm_image(p))
        elif p.type == PartType.input_audio: raise ValueError("OpenAI Responses API does not support audio input; Coming Soon.") 
        elif p.type == PartType.input_video: raise ValueError("OpenAI Responses API does not support video input")
        elif p.type == PartType.input_file:  parts.append(denorm_file(p))
    return dict(type='message', role='user', content=parts)

#### Image

In [ ]:
#| export
def denorm_image(p): return {"type": "input_image", "image_url": p.text}

Image URL

In [ ]:
img_url = "https://img.freepik.com/free-photo/mountain-range-body-water_53876-139760.jpg?semt=ais_hybrid&w=740&q=80"
msg1 = Msg(role='user', content=[Part(type=PartType.input_image, text=img_url), Part(type=PartType.text, text='What is this image?')])

In [ ]:
resp = await oai_cli.responses.create_response(model='gpt-4o-mini', input=denorm_msgs([msg1]), stream=True)
async for comp in acollect_stream(resp, vendor_name=vnd_nm): pass
comp

<div class="prose" markdown="1">

This image depicts a tranquil landscape featuring a lake surrounded by lush trees and mountains in the background. The scene has a calm atmosphere, with reflections of the trees and mountains visible in the water. A wooden deck or platform is visible in the foreground, adding depth to the composition.

<details markdown='1'>

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=25513, completion_tokens=56, total_tokens=25569, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 25513, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 56, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 25569})`

</details>

</div>

Image data

In [ ]:
img_b64 = "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAQAAAAECAIAAAAmkwkpAAAAEElEQVR4nGP4z8AARwzEcQCukw/x0F8jngAAAABJRU5ErkJggg=="
msg1 = Msg(role='user', content=[Part(type=PartType.input_image, text=img_b64), Part(type=PartType.text, text='What color is this pixel?')])

In [ ]:
resp = await oai_cli.responses.create_response(model='gpt-4o-mini', input=denorm_msgs([msg1]), stream=True)
async for comp in acollect_stream(resp, vendor_name=vnd_nm): pass
comp

<div class="prose" markdown="1">

The color of the pixel is pure red.

<details markdown='1'>

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=8513, completion_tokens=10, total_tokens=8523, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 8513, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 10, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 8523})`

</details>

</div>

#### Audio

https://developers.openai.com/api/docs/guides/migrate-to-responses#responses-benefits - responses api support coming soon

#### Files

OpenAI requires a filename, Anthropic requires the media type and Gemini requires the mime type

In [ ]:
#| export
def denorm_file(p):
    if (b64:=data_url(p.text)): return {"type": "input_file", "file_data": p.text, "filename": f"upload.{b64[0].split('/')[-1]}"}
    return {"type": "input_file", "file_url": p.text}

In [ ]:
pdf_url = "https://arxiv.org/pdf/1706.03762"
msg1 = Msg(role='user', content=[Part(type=PartType.input_file, text=pdf_url), Part(type=PartType.text, text='What is the title of this paper?')])

In [ ]:
resp = await oai_cli.responses.create_response(model='gpt-4o-mini', input=denorm_msgs([msg1]), stream=True)
async for comp in acollect_stream(resp, vendor_name=vnd_nm): pass
comp

<div class="prose" markdown="1">

The title of the paper is "Attention Is All You Need."

<details markdown='1'>

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=52889, completion_tokens=14, total_tokens=52903, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 52889, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 14, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 52903})`

</details>

</div>

### Media Tool Call Results

A tool call can return an `input_image` or `input_file`

In [ ]:
#| export
def denorm_tool_result(m:Part):
    "Convert canonical tool result back to OpenAI Responses function_call_output item."
    cid = _sanid(m.data.get('id', '') or m.data.get('call_id'))
    if isinstance(m.text, list):
        out = []
        for p in m.text:
            if   p.type == PartType.text:        out.append({"type": "input_text", "text": p.text or ""})
            elif p.type == PartType.input_image: out.append(denorm_image(p))
            elif p.type == PartType.input_file:  out.append(denorm_file(p))
            else: raise ValueError(f"OpenAI Responses tool_result does not support {p.type}")
        return dict(type='function_call_output', call_id=cid, output=out)
    return dict(type='function_call_output', call_id=cid, output=str(m.text))

In [ ]:
msgs = [Msg(role='user', content=[Part(type=PartType.text, text="What's in this image?")]),
        Msg(role='assistant', content=[Part(type=PartType.tool_use, data={'id': '_test', 'name': 'test_img', 'arguments': {}, 'server': False})]),
        Msg(role='tool', content=[Part(type=PartType.tool_result, text=[Part(type=PartType.input_image, text=img_b64)], data={'id': '_test', 'name': 'test_img'})])]

In [ ]:
resp = await oai_cli.responses.create_response(model='gpt-4o-mini', input=denorm_msgs(msgs), stream=True)
async for comp in acollect_stream(resp, vendor_name=vnd_nm): pass
comp

<div class="prose" markdown="1">

The image appears to be a solid red color.

<details markdown='1'>

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=8529, completion_tokens=11, total_tokens=8540, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 8529, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 11, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 8540})`

</details>

</div>

## Payload

Function to create the payload:

In [ ]:
#| export
@delegates(payload_kwargs)
def mk_payload(msgs, model, **kwargs):
    payload = dict(model=model, input=denorm_msgs(msgs))
    if stream:=kwargs.get('stream'):        payload['stream'] = True
    if sp:=kwargs.get('system'):            payload['instructions'] = denorm_system(sp)
    if mt:=kwargs.get('max_tokens'):        payload['max_output_tokens'] = mt
    if tools:=kwargs.get('tools'):          payload['tools'] = denorm_tool_schs(tools)
    if tchc:=kwargs.get('tool_choice'):     payload['tool_choice'] = denorm_tool_choice(tchc)
    if thk:=kwargs.get('reasoning_effort'): payload['reasoning'] = denorm_reasoning(thk)
    if (wopts:=kwargs.get('web_search_options')) is not None: 
        payload.setdefault('tools', []).append(denorm_web_search(wopts))
    if (temp:=kwargs.get('temperature')) is not None: payload['temperature'] = temp
    return payload

Function to create the default headers:

In [ ]:
#| export
def get_hdrs(api_key=None):
    return {"Authorization": f"Bearer {get_api_key(api_key, 'OPENAI_API_KEY')}"}

## Cost

A function to calculate api Completion cost from raw usage data, and model metadata returned by `get_model_info`

In [ ]:
#| export
def cost(usage, m):
    raw = usage.raw
    cached = raw.get('input_tokens_details', {}).get('cached_tokens', 0)
    in_txt = raw['input_tokens'] - cached
    cost  = in_txt * m.input_cost_per_token + raw['output_tokens'] * m.output_cost_per_token
    cost += cached * m.get('cache_read_input_token_cost', 0)
    return cost

## Register API

Register the final API namespace:

- `norm_tool_calls`, `norm_parts`, `norm_finish`, `norm_usage` required during the Completion object construction.
- `mk_payload` is used to construct the api request payload
- `accollect_stream` is used in streaming api call path
- `get_hdrs` is used to create the deafult request headers
- `op_path` define the non-streaming and streaming OpenAPIClient op names

In [ ]:
#| export
api_ns = dict(norm_tool_calls=norm_tool_calls,
                norm_parts=norm_parts,
                norm_finish=norm_finish,
                norm_usage=norm_usage,
                acollect_stream=acollect_stream,
                mk_payload=mk_payload,
                cost=cost,
                get_hdrs=get_hdrs,
                op_path=('responses.create_response','responses.create_response'))
api_registry.register('openai', **api_ns)

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()